# Silver & Golden Records — OTIF Intelligence on Azure Databricks

**Runs in:** Azure Databricks (Spark SQL). Attach to a Unity Catalog-enabled cluster or SQL Warehouse that can see the **`otif-demo-data-products`** catalog (the SAP S/4HANA supply-chain data products, published to this Azure Databricks workspace via Delta Sharing).

**Purpose:** Reproduce the OTIF golden records — the `silver_otif_production` view, the materialized `silver_otif_production_tbl` table, and the `gold_otif_risk` table — that already exist in the SAP-side Databricks catalog `bdc_otif_demo.otif_demo`, but adapted to the **different source-catalog layout** of this Azure workspace. Here the golden records are written to the existing managed catalog **`otif-demo-bdc`**.

**The key constraint — read vs. write catalogs are different**

`otif-demo-data-products` is a **Delta Sharing catalog** (`CATALOG_DELTASHARING`). It is **read-only**: you can `SELECT` from its tables, but you **cannot** `CREATE SCHEMA`, `CREATE VIEW`, or `CREATE TABLE` inside it. Attempting to do so fails with:

> `INVALID_PARAMETER_VALUE.PRIVILEGE_NOT_APPLICABLE_TO_ENTITY … Privilege CREATE SCHEMA is not applicable to this entity [otif-demo-data-products:CATALOG/CATALOG_DELTASHARING]`

This mirrors the SAP-side integration query exactly: there the shared data products live in per-product `bdc_scdp_*` / `bdc_mdp_*` catalogs (read-only shares), and the golden records are written to a **separate managed catalog** (SAP-side `bdc_otif_demo`). We do the same here, writing into the pre-existing managed catalog **`otif-demo-bdc`**.

| | Role | Catalog on this Azure workspace | Writable? |
|---|---|---|---|
| **Source** (read) | SAP data products via Delta Share | `` `otif-demo-data-products` `` (one schema per data product) | ❌ read-only |
| **Target** (write) | Golden OTIF records | `` `otif-demo-bdc` `` (existing managed catalog) → schema `otif_demo` | ✅ |

**What is different between the two workspaces**

| | SAP Databricks (source of the logic) | This Azure Databricks (target) |
|---|---|---|
| Data-product layout | **One catalog per data product** (`bdc_scdp_productionorder`, `bdc_scdp_workcenter`, …), each with a schema of the same name | **One Delta-share catalog** `otif-demo-data-products` holding **every data product as a schema** (`productionorder`, `workcenter`, …) |
| Production order operation | `bdc_scdp_productionorder.productionorder.productionorderoperation` | `` `otif-demo-data-products`.productionorder.productionorderoperation `` |
| Work center | `bdc_scdp_workcenter.workcenter.workcenter` | `` `otif-demo-data-products`.workcenter.workcenter `` |
| Golden output home | managed catalog `bdc_otif_demo`, schema `otif_demo` | existing managed catalog `` `otif-demo-bdc` ``, schema `otif_demo` |

**Note on catalog names:** both `otif-demo-data-products` (source) and `otif-demo-bdc` (target) contain hyphens, so **neither** is a valid bare SQL identifier. Both must be **back-ticked** (`` `otif-demo-data-products` ``, `` `otif-demo-bdc` ``) everywhere they appear.

The column names, join keys, anomaly formulas and risk tiers are carried over **unchanged** from `src/sql/original-dbx/otif_intelligence_queries.sql` — only the source/target addressing (share catalog + hyphen back-ticking) is rewritten for the new layout.

## Catalog structure (source vs. target)

Two catalogs are involved. The **source** is the read-only Delta-share catalog holding the SAP data products; the **target** is the existing managed catalog `otif-demo-bdc` where the golden records land.

```
otif-demo-data-products          ◄── SOURCE  (Delta Sharing catalog — READ ONLY)
├── customer
├── deliverymgmtconfigurationdata
├── enterpriseresourcecapacity
├── information_schema
├── inventorymgmtconfigurationdata
├── physicalinventorydocument
├── productconfigurationcodes
├── productionorder                 ◄── read: productionorder, productionorderoperation
├── productionorderconfirmation
├── productionrouting
├── productionversion
├── purchaseorder
├── reservationdocument
├── salesorder
├── shippingpoint
├── storagelocation
└── workcenter                      ◄── read: workcenter

otif-demo-bdc                    ◄── TARGET  (existing managed catalog — WRITABLE)
└── otif_demo   (created below)
    ├── silver_otif_production       ◄── write: view
    ├── silver_otif_production_tbl   ◄── write: materialized table
    └── gold_otif_risk               ◄── write: table
```

Because the golden objects go into `otif-demo-bdc` (a normal managed catalog that already exists), all the `CREATE SCHEMA` / `CREATE VIEW` / `CREATE TABLE` statements succeed — they never touch the read-only share.

## Step 0 — Set the target catalog active

The golden records are written into the **existing** managed catalog `` `otif-demo-bdc` `` — it is already provisioned, so there is **no `CREATE CATALOG`** step here (and none of the earlier managed-storage-location issues apply). We just set it active so the unqualified golden objects resolve there, while still fully qualifying every **source** table against `` `otif-demo-data-products` `` so the read/write split stays explicit.

Because `otif-demo-bdc` contains hyphens, it must be back-ticked in `USE CATALOG`.

In [0]:
# The golden records are written into the EXISTING managed catalog `otif-demo-bdc`
# (NOT the read-only share, and NOT a catalog we create here). Set it active so the
# unqualified golden objects resolve there. Hyphenated name -> must be back-ticked.
spark.sql("USE CATALOG `otif-demo-bdc`")
print("active catalog ->", spark.sql("SELECT current_catalog()").first()[0])

active catalog -> otif-demo-bdc


## Step 1 — Verify the source data products are visible

Before building anything, confirm the three schemas the OTIF logic depends on exist in this catalog and hold rows. If any count comes back `0` or the query errors, the notebook is attached to compute that cannot see the catalog (attach a Unity Catalog cluster / SQL Warehouse) — do not proceed, the golden objects would silently be empty.

In [0]:
%sql
SELECT 'productionorder'          AS table_name, COUNT(*) AS row_count
  FROM `otif-demo-data-products`.productionorder.productionorder
UNION ALL
SELECT 'productionorderoperation',              COUNT(*)
  FROM `otif-demo-data-products`.productionorder.productionorderoperation
UNION ALL
SELECT 'workcenter',                            COUNT(*)
  FROM `otif-demo-data-products`.workcenter.workcenter
ORDER BY row_count DESC;

table_name,row_count
productionorderoperation,152196
productionorder,75884
workcenter,986


## Step 2 — Create the golden output schema

The golden records live in a dedicated `otif_demo` schema **inside the existing managed `otif-demo-bdc` catalog** — matching the SAP-side `bdc_otif_demo.otif_demo` layout. This is the fix for the earlier `PRIVILEGE_NOT_APPLICABLE_TO_ENTITY` error: the schema is created in the writable managed catalog, **not** in the read-only Delta share. `IF NOT EXISTS` makes the notebook re-runnable. The hyphenated catalog name is back-ticked.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `otif-demo-bdc`.otif_demo
  COMMENT 'Golden OTIF records: harmonized silver layer + gold risk scores derived from BDC data products.';

## Step 3 — Silver layer: `silver_otif_production` view

The harmonized **silver** view. It joins the production-order header to its operations and enriches each operation with work-center master data, then derives the per-operation anomaly signals (scrap %, work-ratio vs. standard, scrap deviation, a root-cause tag, and a 0–100 anomaly score).

Logic is identical to the SAP-side view; the changes are only in addressing:
- **Write target** is `` `otif-demo-bdc`.otif_demo.silver_otif_production `` (existing managed catalog, back-ticked).
- **Source reads** come from the share: `productionorder` header/operation from the `productionorder` schema, `workcenter` from the `workcenter` schema — all under `` `otif-demo-data-products` ``.

The work-center join is a `LEFT JOIN` on `WorkCenterInternalID` + `Plant` so operations without a matching master record still survive.

In [0]:
%sql
CREATE OR REPLACE VIEW `otif-demo-bdc`.otif_demo.silver_otif_production AS
SELECT
  -- Order header
  po.ProductionOrder,
  po.BillOfOperationsMaterial           AS Material,
  po.ProductionPlant                    AS Plant,
  po.ProductionOrderType,
  po.MRPController,
  po.ProductionSupervisor,

  -- Schedule
  po.OrderScheduledStartDate,
  po.OrderScheduledEndDate,
  po.OrderActualStartDate,
  po.OrderActualEndDate,
  po.OrderConfirmedEndDate,
  DATEDIFF(po.OrderActualEndDate, po.OrderScheduledEndDate)  AS schedule_delay_days,
  CASE
    WHEN po.OrderActualEndDate IS NULL                       THEN 'In Progress'
    WHEN po.OrderActualEndDate > po.OrderScheduledEndDate    THEN 'Late'
    WHEN po.OrderActualEndDate <= po.OrderScheduledEndDate   THEN 'On Time'
    ELSE 'Unknown'
  END                                                        AS delivery_status,

  -- Order quantities
  po.OrderPlannedTotalQty,
  po.OrderConfirmedYieldQty,
  po.OrderConfirmedScrapQty,

  -- Operation
  op.ProductionOrderOperation,
  op.ProductionOrderOperationText,
  op.WorkCenterInternalID,
  wc.WorkCenter                                              AS WorkCenterName,
  wc.WorkCenterCategoryCode,
  wc.MachineType,

  -- Planned
  op.OpPlannedTotalQuantity,
  op.OpPlannedYieldQuantity,
  op.OperationScrapPercent                                   AS PlannedScrapPct,
  op.WorkCenterStandardWorkQty1                              AS StandardWorkQty,
  op.WorkQuantityUnit1                                       AS WorkQtyUnit,

  -- Actual (already on the operation row — no routing join needed)
  op.OpTotalConfirmedYieldQty                                AS ActualYieldQty,
  op.OpTotalConfirmedScrapQty                                AS ActualScrapQty,
  op.OperationConfirmedReworkQty                             AS ActualReworkQty,
  op.ConfirmedWorkQuantity1                                  AS ActualWorkQty,

  -- Anomaly signals
  CASE WHEN op.OpPlannedTotalQuantity > 0
       THEN ROUND(op.OpTotalConfirmedScrapQty
                  / op.OpPlannedTotalQuantity * 100, 2)
       ELSE 0 END                                            AS actual_scrap_pct,

  CASE WHEN op.WorkCenterStandardWorkQty1 > 0
       THEN ROUND(op.ConfirmedWorkQuantity1
                  / op.WorkCenterStandardWorkQty1, 3)
       ELSE NULL END                                         AS work_ratio_vs_standard,

  CASE WHEN op.OpPlannedTotalQuantity > 0
       THEN ROUND(
         (op.OpTotalConfirmedScrapQty / op.OpPlannedTotalQuantity * 100)
         - op.OperationScrapPercent, 2)
       ELSE NULL END                                         AS scrap_deviation_pct,

  -- Root cause per operation
  CASE
    WHEN op.ConfirmedWorkQuantity1
         / NULLIF(op.WorkCenterStandardWorkQty1, 0) > 1.5    THEN 'Work Overrun'
    WHEN op.OpTotalConfirmedScrapQty
         / NULLIF(op.OpPlannedTotalQuantity, 0) > 0.30       THEN 'High Scrap'
    WHEN op.WorkCenterInternalID = '10000040'
     AND op.OpTotalConfirmedScrapQty
         / NULLIF(op.OpPlannedTotalQuantity, 0) > 0.10       THEN 'Final Acceptance Failure'
    ELSE 'Normal'
  END                                                        AS root_cause_tag,

  -- Operation anomaly score 0-100
  LEAST(100, CAST(ROUND(
    LEAST(50, GREATEST(0,
      (COALESCE(op.ConfirmedWorkQuantity1
        / NULLIF(op.WorkCenterStandardWorkQty1,0), 1.0) - 1.0) * 50))
    + LEAST(50, GREATEST(0,
      COALESCE(op.OpTotalConfirmedScrapQty
        / NULLIF(op.OpPlannedTotalQuantity,0) * 100, 0)))
  , 0) AS INT))                                              AS operation_anomaly_score,

  -- Dates
  op.OpActualExecutionStartDate,
  op.OpActualExecutionEndDate

FROM `otif-demo-data-products`.productionorder.productionorder po
INNER JOIN `otif-demo-data-products`.productionorder.productionorderoperation op
  ON po.ProductionOrder = op.ProductionOrder
LEFT  JOIN `otif-demo-data-products`.workcenter.workcenter wc
  ON op.WorkCenterInternalID = wc.WorkCenterInternalID
 AND op.Plant                = wc.Plant;

### Sanity check — silver view

Row count, delivery-status split, and anomaly-score distribution. Use this to confirm the view populated and the anomaly logic is producing a sensible spread before building the gold layer.

In [0]:
%sql
SELECT
  delivery_status,
  root_cause_tag,
  COUNT(*)                               AS rows,
  COUNT(DISTINCT ProductionOrder)        AS orders,
  ROUND(AVG(operation_anomaly_score), 1) AS avg_anomaly_score,
  ROUND(AVG(actual_scrap_pct), 2)        AS avg_scrap_pct
FROM `otif-demo-bdc`.otif_demo.silver_otif_production
GROUP BY delivery_status, root_cause_tag
ORDER BY rows DESC;

delivery_status,root_cause_tag,rows,orders,avg_anomaly_score,avg_scrap_pct
In Progress,Normal,136506,68159,0.0,0.02
On Time,Normal,14580,7654,0.8,0.77
On Time,High Scrap,525,523,44.3,44.98
On Time,Final Acceptance Failure,291,291,20.5,20.58
Late,Normal,122,64,0.2,0.17
In Progress,High Scrap,102,100,43.2,45.13
In Progress,Final Acceptance Failure,59,59,20.6,20.69
Late,High Scrap,5,5,47.2,47.14
On Time,Work Overrun,3,2,53.3,3.33
In Progress,Work Overrun,3,1,52.0,2.00


## Step 4 — Gold layer: `gold_otif_risk` table

The **gold** table rolls the per-operation silver rows up to **one row per production order**, and computes the order-level OTIF risk:
- `otif_risk_score` (0–100): schedule delay (capped) + a weighted share of the worst operation anomaly.
- `is_at_risk`: predictive flag for open orders that already show a live anomaly.
- `risk_tier`: display bucket (`Breached` / `High Risk` / `Medium Risk` / `Low Risk` / `Completed` / `Normal`).
- `primary_root_cause`: most-severe root-cause signal on the order.
- `anomalous_work_centers`: the work centers driving the risk (score ≥ 30).

Materialized as a Delta **table** (not a view) in `` `otif-demo-bdc`.otif_demo `` so it can later be added to a Delta Share and published back to SAP BDC. It reads from the silver view (also in `otif-demo-bdc`), so no share addressing is involved here.

In [0]:
%sql
CREATE OR REPLACE TABLE `otif-demo-bdc`.otif_demo.gold_otif_risk AS
WITH order_anomaly AS (
  SELECT
    ProductionOrder,
    Material,
    Plant,
    ProductionOrderType,
    MRPController,
    ProductionSupervisor,
    OrderScheduledEndDate,
    OrderActualEndDate,
    schedule_delay_days,
    delivery_status,
    OrderPlannedTotalQty,
    OrderConfirmedYieldQty,
    OrderConfirmedScrapQty,

    COUNT(ProductionOrderOperation)                    AS total_operations,
    MAX(operation_anomaly_score)                       AS max_anomaly_score,
    ROUND(AVG(actual_scrap_pct), 2)                    AS avg_scrap_pct,
    ROUND(MAX(actual_scrap_pct), 2)                    AS max_scrap_pct,
    ROUND(MAX(work_ratio_vs_standard), 3)              AS max_work_ratio,
    SUM(ActualScrapQty)                                AS total_scrap_qty,

    -- Anomalous work centers on this order (score >= 30 threshold)
    ARRAY_JOIN(ARRAY_SORT(COLLECT_SET(
      CASE WHEN operation_anomaly_score >= 30 THEN WorkCenterName END
    )), ', ')                                          AS anomalous_work_centers,

    -- Primary root cause: most severe signal wins
    CASE
      WHEN MAX(CASE WHEN root_cause_tag = 'Work Overrun'
                    THEN operation_anomaly_score ELSE 0 END) >= 50 THEN 'Work Overrun'
      WHEN MAX(CASE WHEN root_cause_tag = 'High Scrap'
                    THEN operation_anomaly_score ELSE 0 END) >= 40 THEN 'High Scrap'
      WHEN MAX(CASE WHEN root_cause_tag = 'Final Acceptance Failure'
                    THEN operation_anomaly_score ELSE 0 END) >= 15 THEN 'Final Acceptance Failure'
      ELSE 'Normal'
    END                                                AS primary_root_cause

  FROM `otif-demo-bdc`.otif_demo.silver_otif_production
  GROUP BY
    ProductionOrder, Material, Plant, ProductionOrderType,
    MRPController, ProductionSupervisor,
    OrderScheduledEndDate, OrderActualEndDate,
    schedule_delay_days, delivery_status,
    OrderPlannedTotalQty, OrderConfirmedYieldQty, OrderConfirmedScrapQty
)
SELECT
  *,

  -- OTIF risk score 0-100
  LEAST(100, CAST(ROUND(
    LEAST(40, GREATEST(0, COALESCE(schedule_delay_days, 0) * 4))
    + (max_anomaly_score * 0.6)
  , 0) AS INT))                                        AS otif_risk_score,

  -- Predictive at-risk flag (open orders with active anomaly)
  CASE
    WHEN delivery_status = 'In Progress'
     AND max_anomaly_score >= 30                       THEN true
    ELSE false
  END                                                  AS is_at_risk,

  -- Risk tier for display
  CASE
    WHEN delivery_status = 'Late'                      THEN 'Breached'
    WHEN delivery_status = 'In Progress'
     AND max_anomaly_score >= 40                       THEN 'High Risk'
    WHEN delivery_status = 'In Progress'
     AND max_anomaly_score >= 20                       THEN 'Medium Risk'
    WHEN delivery_status = 'In Progress'
     AND max_anomaly_score >  0                        THEN 'Low Risk'
    WHEN delivery_status = 'On Time'                   THEN 'Completed'
    ELSE 'Normal'
  END                                                  AS risk_tier

FROM order_anomaly;

num_affected_rows,num_inserted_rows


### Sanity check — gold table

Risk-tier × root-cause breakdown with average risk score, total scrap, and the at-risk count. This is the headline summary the OTIF risk product exposes.

In [0]:
%sql
SELECT
  risk_tier,
  primary_root_cause,
  COUNT(*)                                AS orders,
  ROUND(AVG(otif_risk_score), 1)          AS avg_risk_score,
  ROUND(SUM(total_scrap_qty), 0)          AS total_scrap,
  COUNT(CASE WHEN is_at_risk THEN 1 END)  AS at_risk_count
FROM `otif-demo-bdc`.otif_demo.gold_otif_risk
GROUP BY risk_tier, primary_root_cause
ORDER BY avg_risk_score DESC;

risk_tier,primary_root_cause,orders,avg_risk_score,total_scrap,at_risk_count
Breached,High Scrap,5,38.4,283,0
High Risk,Work Overrun,1,34.0,6,1
Completed,Work Overrun,2,33.0,1,0
High Risk,High Scrap,65,29.9,2821,65
Completed,High Scrap,380,29.5,16609,0
Breached,Normal,59,24.8,9,0
Medium Risk,Normal,38,18.8,1039,36
Medium Risk,Final Acceptance Failure,33,17.1,946,28
Completed,Final Acceptance Failure,164,16.9,4763,0
Low Risk,Normal,79,5.2,990,0


## Step 5 — Materialize silver as a Delta table

The silver layer is a **view** (always recomputed from the live Delta-share data products). To share it or query it as a stable snapshot, materialize it into a Delta **table** — `` `otif-demo-bdc`.otif_demo.silver_otif_production_tbl `` — matching the third object in the SAP-side `otif_demo` schema.

In [0]:
%sql
CREATE OR REPLACE TABLE `otif-demo-bdc`.otif_demo.silver_otif_production_tbl AS
SELECT * FROM `otif-demo-bdc`.otif_demo.silver_otif_production;

num_affected_rows,num_inserted_rows


## Step 6 — Verify the golden records exist

Final confirmation that all three objects landed in the `` `otif-demo-bdc`.otif_demo `` schema — the same three objects shown in the SAP-side `bdc_otif_demo.otif_demo` (`gold_otif_risk`, `silver_otif_production`, `silver_otif_production_tbl`).

In [0]:
%sql
SHOW TABLES IN `otif-demo-bdc`.otif_demo;

database,tableName,isTemporary
otif_demo,gold_otif_risk,false
otif_demo,silver_otif_production,false
otif_demo,silver_otif_production_tbl,false


## (Optional) Step 7 — Publish the golden records back to SAP BDC via Delta Sharing

This mirrors the closing section of `otif_intelligence_queries.sql`, which shares the gold + materialized-silver tables back to the SAP Business Data Cloud recipient. It is **optional and environment-specific** — it requires Delta Sharing to be configured on this Azure workspace and a valid BDC recipient. Leave it commented until the recipient name is confirmed.

**Why this cell is Python, not `%sql`:** the pure-SQL `ALTER SHARE ... ADD TABLE` is **not idempotent** — it errors with *"table already in share"* on a second run, and the inverse `REMOVE TABLE` errors on the first run when the table isn't there yet. To make Step 7 safe to re-run (like Steps 1–6), this cell reads `SHOW ALL IN SHARE` first and only adds the tables that are missing. `CREATE SHARE IF NOT EXISTS` and `GRANT` are already idempotent on their own.

**Note on the hyphenated catalog:** `otif-demo-bdc` must be back-ticked inside the `ALTER SHARE ... ADD TABLE` statement. `SHOW ALL IN SHARE` returns names unquoted, so the presence check compares against the unquoted form.

> ⚠️ Sharing publishes data to an external recipient. Only run this once the recipient (`sap-business-data-cloud` or your BDC recipient name) is confirmed and you are authorized to egress this data. The whole cell is commented — uncomment to publish.

In [0]:
# Idempotent Delta Share publish-back — safe to re-run. Uncomment to publish.
#
# CATALOG = "otif-demo-bdc"         # existing managed catalog holding the golden records (NOT the read-only share)
# SCHEMA  = "otif_demo"
# SHARE   = "otif_intelligence_share"
# RECIPIENT = "sap-business-data-cloud"   # confirm your BDC recipient name first
#
# # Hyphenated catalog -> back-tick in SQL; SHOW ALL IN SHARE lists names unquoted.
# CATALOG_SQL = f"`{CATALOG}`"
#
# # 1. Create the share (idempotent).
# spark.sql(
#     f"CREATE SHARE IF NOT EXISTS {SHARE} "
#     f"COMMENT 'OTIF risk scores and anomaly signals generated on Azure Databricks - consumed in SAP BDC'"
# )
#
# # 2. Add only the tables not already in the share (ADD TABLE is not re-runnable on its own).
# shared = {row["name"] for row in spark.sql(f"SHOW ALL IN SHARE {SHARE}").collect()}
# TABLES = {
#     "gold_otif_risk":             "Demo OTIF risk scores per production order",
#     "silver_otif_production_tbl": "Harmonized silver layer - BDC data products joined with anomaly signals",
# }
# for table, comment in TABLES.items():
#     fqn = f"{CATALOG_SQL}.{SCHEMA}.{table}"        # back-ticked catalog for the ALTER SHARE statement
#     shared_name = f"{CATALOG}.{SCHEMA}.{table}"    # SHOW ALL IN SHARE lists names unquoted
#     if shared_name in shared:
#         print(f"already shared: {shared_name}")
#     else:
#         spark.sql(f"ALTER SHARE {SHARE} ADD TABLE {fqn} COMMENT '{comment}'")
#         print(f"added to share: {shared_name}")
#
# # 3. Grant the BDC recipient access (GRANT is idempotent).
# spark.sql(f"GRANT SELECT ON SHARE {SHARE} TO RECIPIENT `{RECIPIENT}`")
#
# # 4. Verify.
# spark.sql(f"SHOW ALL IN SHARE {SHARE}").show(truncate=False)